# Masai Interactive: Q1 deck from Arguments — PowerPoint + Google Slides

## What this shows
Take the same `Argument` objects that drive PDFs in `reports/01_charts_and_pdf.ipynb` and render a 5-slide branded deck under Masai Interactive's colors. PowerPoint output runs inline; Google Slides output is shown as call shape (requires a Google service account).

## Why it matters
Masai Interactive's client deliverables are decks, not PDFs. The `Argument` model means the same analysis flows into whichever surface a client wants — PDF, PPTX, or Google Slides — without rewriting the analysis layer. One upstream, many downstream.

## Prereqs
- `pip install 'siege-utilities[reporting,pptx]'` (python-pptx + matplotlib).
- Optional: a Google service-account JSON for the live Slides path.

## Next
- `analytics/02_ga_end_to_end.ipynb` — Masai's weekly GA digest, one-page PDF.


## 1. Prepare a chart image under Masai branding

Same approach as `reports/01` — render a matplotlib chart using Masai Interactive's primary color, save as PNG, hand off to PowerPointGenerator as an image path.

In [1]:
%matplotlib inline
from pathlib import Path
import matplotlib.pyplot as plt
from siege_utilities.reporting.client_branding import ClientBrandingManager

brand = ClientBrandingManager().branding_templates['masai_interactive']
out = Path('output'); out.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    ['organic', 'paid', 'direct', 'referral'],
    [42, 28, 18, 12],
    color=brand['colors']['primary'],
)
ax.set_title('Client X — channel mix (% of sessions)',
             color=brand['colors']['primary'])
fig.tight_layout()
chart_path = out / 'masai_channel_mix.png'
fig.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'chart: {chart_path}')


chart: output/masai_channel_mix.png


## 2. Assemble a 5-slide deck

`PowerPointGenerator.create_comprehensive_presentation` sets up title slide + branded header / footer from the Masai template. `add_chart_slide` / `add_text_slide` / `add_table_slide` append sections in whatever narrative order fits.

In [2]:
from siege_utilities.reporting.powerpoint_generator import PowerPointGenerator

pp = PowerPointGenerator(client_name='masai_interactive', output_dir=out)
deck = pp.create_comprehensive_presentation(
    title='Client X — Q1 digital performance',
    author='Masai Interactive',
)
pp.add_text_slide(
    deck, 'Executive summary',
    'Site sessions up 22% quarter-over-quarter; organic channel drove most of the gain.',
)
pp.add_chart_slide(
    deck, 'Channel mix', charts=[str(chart_path)],
    description='Sessions distributed across acquisition channels in Q1.',
)
pp.add_table_slide(
    deck, 'Top 3 pages',
    table_data=[
        {'page': '/', 'sessions': 12500, 'conv_rate_pct': 3.4},
        {'page': '/pricing', 'sessions': 4200, 'conv_rate_pct': 5.8},
        {'page': '/blog/2026-trends', 'sessions': 3100, 'conv_rate_pct': 0.9},
    ],
)
pp.add_summary_slide(
    deck, 'Next quarter',
    key_points=['double-down on organic content pipeline'],
)
print('deck sections:', sorted(deck.keys()))


deck sections: ['metadata', 'sections', 'slide_structure', 'title', 'type']


## 3. Generate the PPTX

`generate_powerpoint_presentation` writes the actual `.pptx` to disk. Masai ships these to the client via email or a shared drive.

In [3]:
pptx_path = out / 'client_x_q1_2026.pptx'
_ok = pp.generate_powerpoint_presentation(deck, output_path=str(pptx_path))
assert _ok, 'PPTX generation failed'
print(f'PPTX written: {pptx_path}')
print(f'size: {Path(pptx_path).stat().st_size:,} bytes')


[siege_utilities] 2026-04-24 17:34:50,940 INFO: PowerPoint presentation generated successfully: output/client_x_q1_2026.pptx


PPTX written: output/client_x_q1_2026.pptx
size: 53,109 bytes


## 4. Google Slides output (call shape)

For clients that prefer Google Workspace, the `google_slides` module consumes the same `Argument` inputs and calls the Slides API directly. Requires a service account JSON. Call shape:

```python
from siege_utilities.analytics.google_slides import GoogleSlidesConnector

slides = GoogleSlidesConnector.from_service_account_file('path/to/svc.json')
presentation_id = slides.create_presentation(
    title='Client X — Q1 digital performance',
    sections=deck['sections'],
    branding=brand,
)
print('https://docs.google.com/presentation/d/' + presentation_id)
```


## Related

- **Source**: `siege_utilities/reporting/powerpoint_generator.py`, `siege_utilities/analytics/google_slides.py`, `siege_utilities/reporting/client_branding.py`.
- **Tests**: `tests/test_survey_render.py` (for the Argument surface), `tests/test_client_branding*.py`.
- **Next**: `analytics/02_ga_end_to_end.ipynb` — Masai's GA digest pipeline in the same style.
